# EUREKA: CRISPR Precision Locus Engineering & Base Editing
### In-Silico Off-Target Cleavage & Single-Nucleotide Transition Analysis

This notebook walks through precision transition mutagenesis with zero double-strand breaks:
1. **Therapeutic Target Selection**: BCL11A, PCSK9, CCR5, and TTR
2. **Deamination Window Activity Profiling**: Positions 4-8 across protospacers
3. **CFD Genome-Wide Specificity Matrix**: Cutting-Frequency Determination score
4. **Guide RNA Secondary Structure MFE**: Nussinov dynamic programming folding
5. **Product Purity & Bystander Indel Profiling**

In [1]:
import sys
sys.path.append('..')
from backend.crispr_locus_engine import LocusSimulator, LocusTargetDefinition

simulator = LocusSimulator()
print('[CRISPR] Initializing Locus Engineering Workbench...')
print('[GENOME] Reference Assembly: GRCh38.p14 / GENCODE v46')
print('[EDITORS] Loaded ABE8e (Adenine) and CBE4max (Cytosine) kinetic models.')

[CRISPR] Initializing Locus Engineering Workbench...
[GENOME] Reference Assembly: GRCh38.p14 / GENCODE v46
[EDITORS] Loaded ABE8e (Adenine) and CBE4max (Cytosine) kinetic models.


## Therapeutic Loci Multi-Target Assessment
Evaluating clinical base-editing targets for hemoglobinopathies, cardiovascular risk, and amyloidosis.

In [2]:
targets = ['BCL11A', 'PCSK9', 'CCR5', 'TTR']
print('=' * 88)
print('  GENE    | CHROMOSOME COORDINATES     | ON-TARGET EFF. | CFD SCORE | PURITY | GUIDE MFE')
print('=' * 88)
for t in targets:
    res = simulator.evaluate_locus(t)
    print(f"  {res['target_locus']:7s} | {res['chromosomal_coordinates']:26s} | {res['on_target_efficiency_pct']:13.1f}% | {res['cfd_off_target_cleavage_score']:9.2f} | {res['product_purity_ratio']:6.3f} | {res['guide_rna_stability']['minimum_free_energy_kcal_mol']:5.1f} kcal/mol")
print('=' * 88)

  GENE    | CHROMOSOME COORDINATES     | ON-TARGET EFF. | CFD SCORE | PURITY | GUIDE MFE
  BCL11A  | chr2:60,495,280-60,495,302 |          84.6% |      0.04 |  0.952 | -22.2 kcal/mol
  PCSK9   | chr1:55,039,812-55,039,834 |          82.3% |      0.06 |  0.938 | -18.5 kcal/mol
  CCR5    | chr3:46,373,450-46,373,472 |          81.9% |      0.05 |  0.941 | -20.1 kcal/mol
  TTR     | chr18:31,592,940-31,592,962|          83.7% |      0.03 |  0.965 | -24.4 kcal/mol


## Off-Target CFD Sensitivity Analysis
Simulating guide mismatch tolerance across seed (positions 11-20) and distal regions.

In [3]:
proto = 'CTAACAGTTGCTTTTATCAC'
cand_distal = 'CAAACAGTTGCTTTTATCAC'
cand_middle = 'CTAACAGATGCTTTTATCAC'
cand_seed = 'CTAACAGTTGCTTTTATCCC'
cand_dual = 'CTAACAGTTGCTTTTAACCC'

print(f'On-Target Protospacer: {proto}')
print(f'Mismatch at pos 2 (Distal): CFD = {simulator.compute_cfd_score(proto, cand_distal):.4f} (Tolerated)')
print(f'Mismatch at pos 8 (Middle): CFD = {simulator.compute_cfd_score(proto, cand_middle):.4f} (Moderate Reduction)')
print(f'Mismatch at pos 16 (Seed):  CFD = {simulator.compute_cfd_score(proto, cand_seed):.4f} (Severe Discrimination)')
print(f'Dual Mismatches (Seed):     CFD = {simulator.compute_cfd_score(proto, cand_dual):.4f} (Negligible Cleavage Risk)')

On-Target Protospacer: CTAACAGTTGCTTTTATCAC
Mismatch at pos 2 (Distal): CFD = 0.7800 (Tolerated)
Mismatch at pos 8 (Middle): CFD = 0.4500 (Moderate Reduction)
Mismatch at pos 16 (Seed):  CFD = 0.2200 (Severe Discrimination)
Dual Mismatches (Seed):     CFD = 0.0484 (Negligible Cleavage Risk)
